# Phase 3 — Transfer Learning

**Learning Goals**
- Load and adapt a pretrained CNN (VGG16, ResNet50, or MobileNetV2)
- Freeze the base and train a custom classification head
- Fine-tune the top layers of the base model
- Compare transfer learning performance vs the baseline MLP

**Tasks to complete:**
1. `TODO 6` in `src/models.py` — `build_transfer_model()`
2. `TODO 7` in `src/models.py` — `unfreeze_top_layers()`

Run `pytest tests/test_phase3.py -v` to check your work.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data_loader import get_data_generators, compute_class_weights
from src.train import get_callbacks, train_model, plot_history
from src.evaluate import evaluate_model

In [2]:
# Use full 224x224 resolution for pretrained models
train_gen, val_gen, test_gen = get_data_generators(
    data_dir='../data/chest_xray',
    img_size=(224, 224),
    batch_size=32,
)
class_weights = compute_class_weights(train_gen)

Found 2914 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


## Step 1 — Feature Extraction (Frozen Base)

Implement `build_transfer_model()` in `src/models.py`, then run the cells below.

In [3]:
from src.models import build_transfer_model

model = build_transfer_model(
    base_model_name='MobileNetV2',
    input_shape=(224, 224, 3),
    dropout_rate=0.3,
    learning_rate=1e-3,
    freeze_base=True,
)
model.summary()

# Count trainable vs non-trainable params
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
frozen    = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
print(f'Trainable params: {trainable:,}  |  Frozen params: {frozen:,}')

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,587,201 (9.87 MB)

 Trainable params: 328,705 (1.25 MB)

 Non-trainable params: 2,258,496 (8.62 MB)

Trainable params: 328,705  |  Frozen params: 2,258,496


In [4]:
callbacks = get_callbacks(model_name='mobilenetv2_frozen', patience=5)

history_frozen = train_model(
    model, train_gen, val_gen,
    epochs=15,
    class_weights=class_weights,
    callbacks=callbacks,
)
plot_history(history_frozen, title='MobileNetV2 — Frozen Base')

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x00000255401D3CE0>

## Step 2 — Fine-Tuning

Implement `unfreeze_top_layers()` in `src/models.py`, then continue training with a lower learning rate.

In [5]:
from src.models import unfreeze_top_layers

model = unfreeze_top_layers(model, num_layers=20)

trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f'Trainable params after unfreezing: {trainable:,}')
print(f'Fine-tune learning rate: {model.optimizer.learning_rate.numpy():.2e}')

Trainable params after unfreezing: 1,534,785
Fine-tune learning rate: 1.00e-05


In [6]:
callbacks_ft = get_callbacks(model_name='mobilenetv2_finetune', patience=5)

history_ft = train_model(
    model, train_gen, val_gen,
    epochs=10,
    class_weights=class_weights,
    callbacks=callbacks_ft,
)
plot_history(history_ft, title='MobileNetV2 — Fine-Tuned')

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x00000255437528E0>

## Step 3 — Compare All Three Base Models

Train VGG16, ResNet50, and MobileNetV2 and compare their test performance.

In [7]:
from src.evaluate import evaluate_model, compare_models

comparison = {}
for base in ['MobileNetV2', 'VGG16', 'ResNet50']:
    print(f'\n--- Training {base} ---')
    m = build_transfer_model(base_model_name=base, input_shape=(224, 224, 3),
                              freeze_base=True, learning_rate=1e-3)
    cb = get_callbacks(model_name=base.lower(), patience=5)
    train_model(m, train_gen, val_gen, epochs=10, class_weights=class_weights, callbacks=cb)
    # TODO: implement evaluate_model() in src/evaluate.py
    comparison[base] = evaluate_model(m, test_gen)
    print(f"{base} test accuracy: {comparison[base]['accuracy']:.4f}")

compare_models(comparison)


--- Training MobileNetV2 ---


UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x00000255402FE1B0>

## 4. Save Best Model

In [8]:
# Save the best performing transfer learning model
best_base = max(comparison, key=lambda k: comparison[k]['accuracy'])
print(f'Best base model: {best_base}')
model.save(f'../models/{best_base.lower()}_best.keras')
print('Model saved.')

ValueError: max() iterable argument is empty

## 5. Reflection Questions

1. Why does feature extraction work even though the base was trained on natural images (ImageNet), not X-rays?
2. Which layers learn low-level vs high-level features? Why do we only fine-tune the top layers?
3. Why must the learning rate be much lower during fine-tuning than during feature extraction?
4. Compare your best transfer model against the Phase 1 MLP. Quantify the improvement.

**Your answers here:**

1. 
2. 
3. 
4. 